In [0]:
from pyspark.sql import SparkSession

In [0]:
data = [
    ("Alice", "HR", 1000),
    ("Bob", "IT", 2000),
    ("Charlie", "HR", 1500),
    ("David", "Finance", 2500),
    ("Eva", "IT", 1800),
    ("Frank", "Finance", 2200)
]

columns = ["name", "department", "salary"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Write DataFrame using partitioning
output_path = "/Workspace/ApacheSparkGuide/OutputData"

df.write \
    .mode("overwrite") \
    .partitionBy("department") \
    .parquet(output_path)                       # convert data into parquet format

## DYNAMIC PARTITION PRUNING

In [0]:
from pyspark.sql.functions import *

In [0]:
# Without Partition = small table = already filtered 
# With Partition = big table = not filtered

df_withoutpart = spark.read.format("parquet") \
                    .load((output_path)) \
                    .filter(col('department') == "HR")              # only read the HR

df_withpart = spark.read.format("parquet") \
                    .load((output_path))

In [0]:
display(df_withoutpart)

display(df_withpart)

name,salary,department
Charlie,1500,HR
Alice,1000,HR


name,salary,department
Charlie,1500,HR
Frank,2200,Finance
Alice,1000,HR
David,2500,Finance
Bob,2000,IT
Eva,1800,IT


In [0]:
# DPP: reduces the amount of data scanned from a large table (fact table) by pruning partitions at runtime based on the values produced by a join with a smalled table

df_join = df_withoutpart.join(df_withpart, (df_withoutpart['name'] == df_withpart['name']) & (df_withoutpart['department'] == df_withpart['department']), 'inner')

display(df_join) 

name,salary,department,name,salary,department
Charlie,1500,HR,Charlie,1500,HR
Alice,1000,HR,Alice,1000,HR
